# 05 — Data Quality Summary (Quality Gate)

This notebook computes **data quality metrics** for Bronze, Silver, and Gold layers.

## Goals
- Reconciliation: row counts and drop rates across layers
- Completeness: null-rate by column (Silver)
- Validity: business sanity rules (Silver)
- Drift: schema differences vs expected columns
- Deterministic, reproducible results for the configured month

## Outputs (Delta)
- `data_quality_summary` (monthly snapshot)
- (Optional) `dq_nulls_by_column` (column-level completeness metrics)

In [1]:
# ==============================
# CONFIG
# ==============================

DATASET = "nyc_tlc_yellow"

BRONZE_PATH = f"../lakehouse/bronze/{DATASET}"
SILVER_PATH = f"../lakehouse/silver/{DATASET}"
GOLD_PATH = f"../lakehouse/gold/{DATASET}"

# Gold table produced in 03_gold_kpis.ipynb
GOLD_KPI_DAILY_PATH = f"{GOLD_PATH}/kpi_daily_overview"

YEAR = 2024
MONTH = 1  # 1-12

# Outputs
DQ_SUMMARY_PATH = f"{GOLD_PATH}/data_quality_summary"
DQ_NULLS_BY_COLUMN_PATH = f"{GOLD_PATH}/dq_nulls_by_column"

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_KPI_DAILY_PATH:", GOLD_KPI_DAILY_PATH)
print("YEAR/MONTH:", YEAR, MONTH)
print("DQ_SUMMARY_PATH:", DQ_SUMMARY_PATH)
print("DQ_NULLS_BY_COLUMN_PATH:", DQ_NULLS_BY_COLUMN_PATH)

BRONZE_PATH: ../lakehouse/bronze/nyc_tlc_yellow
SILVER_PATH: ../lakehouse/silver/nyc_tlc_yellow
GOLD_KPI_DAILY_PATH: ../lakehouse/gold/nyc_tlc_yellow/kpi_daily_overview
YEAR/MONTH: 2024 1
DQ_SUMMARY_PATH: ../lakehouse/gold/nyc_tlc_yellow/data_quality_summary
DQ_NULLS_BY_COLUMN_PATH: ../lakehouse/gold/nyc_tlc_yellow/dq_nulls_by_column


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("lakehouse-data-quality")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

26/02/24 11:43:39 WARN Utils: Your hostname, willian-A320M-S2H resolves to a loopback address: 127.0.1.1; using 192.168.0.110 instead (on interface enp6s0)
26/02/24 11:43:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/willian/PhaifferTech/nyc-tlc-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/willian/.ivy2/cache
The jars for the packages stored in: /home/willian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-40e150f3-b934-4608-9060-6d6d8b33e399;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 170ms :: artifacts dl 11ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |  

In [3]:
from pyspark.sql import functions as F
import os
import shutil

def month_window(year: int, month: int):
    """
    Returns (start_date_expr, end_date_expr) where end is exclusive.
    """
    start = F.to_date(F.lit(f"{year:04d}-{month:02d}-01"))
    end = F.add_months(start, 1)
    return start, end

def safe_rate(numerator_col, denominator_col):
    """
    Returns a safe rate expression avoiding division by zero.
    """
    return F.when(denominator_col == 0, F.lit(0.0)).otherwise(numerator_col / denominator_col)

def assert_non_null(value, msg: str):
    assert value is not None, msg

In [4]:
# Load layers
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
silver_df = spark.read.format("delta").load(SILVER_PATH)
gold_kpi_df = spark.read.format("delta").load(GOLD_KPI_DAILY_PATH)

# Row counts
bronze_rows = bronze_df.count()
silver_rows = silver_df.count()
gold_days = gold_kpi_df.count()

print(f"[Bronze] rows: {bronze_rows}")
print(f"[Silver] rows: {silver_rows}")
print(f"[Gold KPI] days (rows): {gold_days}")

# Silver pickup_date bounds
silver_bounds = silver_df.select(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),
).collect()[0]

print("[Silver] pickup_date min:", silver_bounds["min_pickup_date"])
print("[Silver] pickup_date max:", silver_bounds["max_pickup_date"])

assert_non_null(silver_bounds["min_pickup_date"], "Silver min pickup_date is null.")
assert_non_null(silver_bounds["max_pickup_date"], "Silver max pickup_date is null.")

26/02/24 11:43:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[Bronze] rows: 2964624
[Silver] rows: 2839382
[Gold KPI] days (rows): 31
[Silver] pickup_date min: 2024-01-01
[Silver] pickup_date max: 2024-01-31


In [5]:
start_date, end_date = month_window(YEAR, MONTH)

window_check = (
    silver_df
    .select(
        F.min("pickup_date").alias("min_pickup_date"),
        F.max("pickup_date").alias("max_pickup_date"),
    )
    .withColumn("expected_start", start_date)
    .withColumn("expected_end_exclusive", end_date)
    .collect()[0]
)

print("[Config] expected_start:", window_check["expected_start"])
print("[Config] expected_end_exclusive:", window_check["expected_end_exclusive"])

assert window_check["min_pickup_date"] >= window_check["expected_start"], \
    f"Silver starts before expected month: {window_check['min_pickup_date']}"
assert window_check["max_pickup_date"] < window_check["expected_end_exclusive"], \
    f"Silver ends after expected month: {window_check['max_pickup_date']}"

26/02/24 11:43:58 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


[Config] expected_start: 2024-01-01
[Config] expected_end_exclusive: 2024-02-01


In [6]:
from pyspark.sql import functions as F

# Diagnose Silver key nulls (these break daily aggregations)
null_pickup_date = silver_df.filter(F.col("pickup_date").isNull()).count()
null_pickup_ts = silver_df.filter(F.col("tpep_pickup_datetime").isNull()).count()
null_dropoff_ts = silver_df.filter(F.col("tpep_dropoff_datetime").isNull()).count()

print("[DQ] Silver null pickup_date:", null_pickup_date)
print("[DQ] Silver null tpep_pickup_datetime:", null_pickup_ts)
print("[DQ] Silver null tpep_dropoff_datetime:", null_dropoff_ts)

# If this is > 0, we must explain why and decide whether to filter in Silver (preferred)

[DQ] Silver null pickup_date: 0
[DQ] Silver null tpep_pickup_datetime: 0
[DQ] Silver null tpep_dropoff_datetime: 0


In [7]:
# Bronze -> Silver drop rate
bronze_rows_lit = F.lit(bronze_rows)
silver_rows_lit = F.lit(silver_rows)

drop_abs = bronze_rows - silver_rows
drop_rate = 0.0 if bronze_rows == 0 else (drop_abs / bronze_rows)

print(f"[Recon] Bronze->Silver dropped rows: {drop_abs}")
print(f"[Recon] Bronze->Silver drop rate: {drop_rate:.6f}")

# Silver -> Gold consistency: sum(trips) should match Silver rows
gold_trip_sum = gold_kpi_df.select(F.sum("trips").alias("sum_trips")).collect()[0]["sum_trips"]
gold_trip_sum = int(gold_trip_sum) if gold_trip_sum is not None else 0

print(f"[Recon] Gold sum(trips): {gold_trip_sum}")
print(f"[Recon] Silver rows: {silver_rows}")

diff = silver_rows - gold_trip_sum

print(f"[Recon] Difference (Silver rows - Gold sum(trips)): {diff}")

if diff != 0:
    print("[Recon] MISMATCH detected. Running diagnostics to find root cause...")
else:
    print("[Recon] OK: Gold sum(trips) matches Silver rows.")

[Recon] Bronze->Silver dropped rows: 125242
[Recon] Bronze->Silver drop rate: 0.042245


[Recon] Gold sum(trips): 2839382
[Recon] Silver rows: 2839382
[Recon] Difference (Silver rows - Gold sum(trips)): 0
[Recon] OK: Gold sum(trips) matches Silver rows.


In [8]:
from pyspark.sql import functions as F

# Compare per-day counts: Silver vs Gold
silver_daily = (
    silver_df
    .groupBy("pickup_date")
    .agg(F.count(F.lit(1)).alias("silver_trips"))
)

gold_daily = (
    gold_kpi_df
    .select("pickup_date", F.col("trips").cast("long").alias("gold_trips"))
)

cmp = (
    silver_daily
    .join(gold_daily, on="pickup_date", how="full_outer")
    .na.fill(0, subset=["silver_trips", "gold_trips"])
    .withColumn("diff_trips", F.col("silver_trips") - F.col("gold_trips"))
    .orderBy(F.abs(F.col("diff_trips")).desc(), F.col("pickup_date").asc())
)

cmp.show(50, truncate=False)

total_diff = cmp.select(F.sum("diff_trips").alias("total_diff")).collect()[0]["total_diff"]
print("[Recon] Total diff from per-day comparison:", total_diff)

+-----------+------------+----------+----------+
|pickup_date|silver_trips|gold_trips|diff_trips|
+-----------+------------+----------+----------+
|2024-01-01 |74830       |74830     |0         |
|2024-01-02 |72341       |72341     |0         |
|2024-01-03 |79188       |79188     |0         |
|2024-01-04 |99066       |99066     |0         |
|2024-01-05 |99019       |99019     |0         |
|2024-01-06 |92874       |92874     |0         |
|2024-01-07 |64582       |64582     |0         |
|2024-01-08 |76897       |76897     |0         |
|2024-01-09 |89441       |89441     |0         |
|2024-01-10 |91525       |91525     |0         |
|2024-01-11 |100972      |100972    |0         |
|2024-01-12 |99428       |99428     |0         |
|2024-01-13 |100493      |100493    |0         |
|2024-01-14 |90036       |90036     |0         |
|2024-01-15 |73961       |73961     |0         |
|2024-01-16 |89264       |89264     |0         |
|2024-01-17 |105460      |105460    |0         |
|2024-01-18 |105876 

In [9]:
# Minimal expected Silver columns for this project (adjust if your Silver contract differs)
expected_silver_cols = {
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_date",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
}

silver_cols = set(silver_df.columns)

missing_cols = sorted(list(expected_silver_cols - silver_cols))
extra_cols = sorted(list(silver_cols - expected_silver_cols))

print("[Drift] Missing expected Silver columns:", missing_cols)
print("[Drift] Extra Silver columns (not in minimal contract):", extra_cols)

[Drift] Missing expected Silver columns: []
[Drift] Extra Silver columns (not in minimal contract): ['Airport_fee', 'DOLocationID', 'PULocationID', 'RatecodeID', 'VendorID', '_dataset', '_ingestion_timestamp', '_source_file', 'congestion_surcharge', 'extra', 'improvement_surcharge', 'mta_tax', 'payment_type', 'store_and_fwd_flag', 'total_amount']


In [10]:
total_rows = silver_rows
print("[Completeness] silver_rows:", total_rows)

null_exprs = []
for c in silver_df.columns:
    null_exprs.append(F.sum(F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0))).alias(c))

null_counts_row = silver_df.agg(*null_exprs).collect()[0].asDict()

nulls_data = []
for c, null_count in null_counts_row.items():
    null_count_int = int(null_count) if null_count is not None else 0
    null_rate = 0.0 if total_rows == 0 else (null_count_int / total_rows)
    nulls_data.append((c, null_count_int, float(null_rate), YEAR, MONTH))

dq_nulls_by_col_df = spark.createDataFrame(
    nulls_data,
    ["column_name", "null_count", "null_rate", "year", "month"]
).orderBy(F.col("null_rate").desc(), F.col("null_count").desc())

dq_nulls_by_col_df.show(50, truncate=False)

[Completeness] silver_rows: 2839382


+---------------------+----------+-------------------+----+-----+
|column_name          |null_count|null_rate          |year|month|
+---------------------+----------+-------------------+----+-----+
|RatecodeID           |115239    |0.04058594440621234|2024|1    |
|passenger_count      |115239    |0.04058594440621234|2024|1    |
|congestion_surcharge |115239    |0.04058594440621234|2024|1    |
|store_and_fwd_flag   |115239    |0.04058594440621234|2024|1    |
|Airport_fee          |115239    |0.04058594440621234|2024|1    |
|fare_amount          |0         |0.0                |2024|1    |
|VendorID             |0         |0.0                |2024|1    |
|extra                |0         |0.0                |2024|1    |
|PULocationID         |0         |0.0                |2024|1    |
|mta_tax              |0         |0.0                |2024|1    |
|tpep_pickup_datetime |0         |0.0                |2024|1    |
|tip_amount           |0         |0.0                |2024|1    |
|improveme

In [11]:
# Compute duration in seconds deterministically
silver_with_duration = (
    silver_df
    .withColumn(
        "trip_duration_sec",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")).cast("long")
    )
)

total_rows_col = F.lit(total_rows)

violations_df = silver_with_duration.select(
    F.sum(F.when((F.col("trip_duration_sec").isNull()) | (F.col("trip_duration_sec") <= 0), 1).otherwise(0)).alias("v_duration_non_positive"),
    F.sum(F.when(F.col("trip_duration_sec") >= 48 * 3600, 1).otherwise(0)).alias("v_duration_too_large"),
    F.sum(F.when(F.col("fare_amount").isNull() | (F.col("fare_amount") < 0), 1).otherwise(0)).alias("v_fare_negative_or_null"),
    F.sum(F.when(F.col("trip_distance").isNull() | (F.col("trip_distance") < 0), 1).otherwise(0)).alias("v_distance_negative_or_null"),
    F.sum(F.when(F.col("passenger_count").isNull() | (F.col("passenger_count") < 1), 1).otherwise(0)).alias("v_passenger_invalid"),
    F.sum(F.when((F.col("pickup_date") < start_date) | (F.col("pickup_date") >= end_date), 1).otherwise(0)).alias("v_outside_month_window"),
).collect()[0].asDict()

for k, v in violations_df.items():
    v_int = int(v) if v is not None else 0
    rate = 0.0 if total_rows == 0 else (v_int / total_rows)
    print(f"[Validity] {k}: {v_int} ({rate:.6f})")

# Optional hard gates (portfolio-grade).
# You can tune thresholds. Here we block if window drift exists.
assert int(violations_df["v_outside_month_window"]) == 0, "Silver contains pickup_date outside configured month window."

[Validity] v_duration_non_positive: 0 (0.000000)
[Validity] v_duration_too_large: 3 (0.000001)
[Validity] v_fare_negative_or_null: 0 (0.000000)
[Validity] v_distance_negative_or_null: 0 (0.000000)
[Validity] v_passenger_invalid: 115239 (0.040586)
[Validity] v_outside_month_window: 0 (0.000000)


In [12]:
run_ts = F.current_timestamp()

summary_rows = []

# Reconciliation metrics
summary_rows.append(("reconciliation", "bronze_rows", float(bronze_rows), None))
summary_rows.append(("reconciliation", "silver_rows", float(silver_rows), None))
summary_rows.append(("reconciliation", "bronze_to_silver_drop_rows", float(drop_abs), None))
summary_rows.append(("reconciliation", "bronze_to_silver_drop_rate", float(drop_rate), None))
summary_rows.append(("reconciliation", "gold_kpi_days", float(gold_days), None))
summary_rows.append(("reconciliation", "gold_sum_trips", float(gold_trip_sum), None))

# Validity metrics
for k, v in violations_df.items():
    v_int = int(v) if v is not None else 0
    rate = 0.0 if total_rows == 0 else (v_int / total_rows)
    summary_rows.append(("validity", k, float(v_int), float(rate)))

# Drift metrics
summary_rows.append(("drift", "missing_expected_silver_columns", float(len(missing_cols)), None))
summary_rows.append(("drift", "extra_silver_columns", float(len(extra_cols)), None))

dq_summary_df = spark.createDataFrame(
    summary_rows,
    ["category", "metric_name", "metric_value", "metric_rate"]
).withColumn("year", F.lit(YEAR)) \
 .withColumn("month", F.lit(MONTH)) \
 .withColumn("run_timestamp", run_ts)

dq_summary_df.orderBy("category", "metric_name").show(200, truncate=False)

+--------------+-------------------------------+-------------------+--------------------+----+-----+--------------------------+
|category      |metric_name                    |metric_value       |metric_rate         |year|month|run_timestamp             |
+--------------+-------------------------------+-------------------+--------------------+----+-----+--------------------------+
|drift         |extra_silver_columns           |15.0               |NULL                |2024|1    |2026-02-24 11:44:11.661074|
|drift         |missing_expected_silver_columns|0.0                |NULL                |2024|1    |2026-02-24 11:44:11.661074|
|reconciliation|bronze_rows                    |2964624.0          |NULL                |2024|1    |2026-02-24 11:44:11.661074|
|reconciliation|bronze_to_silver_drop_rate     |0.04224549217708552|NULL                |2024|1    |2026-02-24 11:44:11.661074|
|reconciliation|bronze_to_silver_drop_rows     |125242.0           |NULL                |2024|1    |2026

In [13]:
# Best-effort vacuum using absolute paths (Spark SQL does not like relative paths)
import os

# Ensure deterministic physical cleanup for local/dev environments
for p in [DQ_SUMMARY_PATH, DQ_NULLS_BY_COLUMN_PATH]:
    if os.path.exists(p):
        shutil.rmtree(p)

(
    dq_summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DQ_SUMMARY_PATH)
)

(
    dq_nulls_by_col_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DQ_NULLS_BY_COLUMN_PATH)
)

print("DQ summary written to:", DQ_SUMMARY_PATH)
print("DQ nulls-by-column written to:", DQ_NULLS_BY_COLUMN_PATH)

abs_summary = os.path.abspath(DQ_SUMMARY_PATH)
abs_nulls = os.path.abspath(DQ_NULLS_BY_COLUMN_PATH)

try:
    spark.sql(f"VACUUM delta.`{abs_summary}`")
    spark.sql(f"VACUUM delta.`{abs_nulls}`")
    print("[DQ] VACUUM executed (default retention).")
except Exception as e:
    print("[DQ] VACUUM skipped due to environment limitation:", str(e))

DQ summary written to: ../lakehouse/gold/nyc_tlc_yellow/data_quality_summary
DQ nulls-by-column written to: ../lakehouse/gold/nyc_tlc_yellow/dq_nulls_by_column


Deleted 0 files and directories in a total of 1 directories.


Deleted 0 files and directories in a total of 1 directories.
[DQ] VACUUM executed (default retention).


In [14]:
dq_summary_check = spark.read.format("delta").load(DQ_SUMMARY_PATH)
dq_nulls_check = spark.read.format("delta").load(DQ_NULLS_BY_COLUMN_PATH)

print("[Postwrite] dq_summary rows:", dq_summary_check.count())
print("[Postwrite] dq_nulls_by_column rows:", dq_nulls_check.count())

dq_summary_check.orderBy("category", "metric_name").show(200, truncate=False)
dq_nulls_check.orderBy(F.col("null_rate").desc()).show(30, truncate=False)

# Hard asserts: ensure summary has expected categories at least
cats = [r["category"] for r in dq_summary_check.select("category").distinct().collect()]
print("[Postwrite] categories:", sorted(cats))

assert "reconciliation" in cats, "Missing reconciliation category in DQ summary."
assert "validity" in cats, "Missing validity category in DQ summary."
assert "drift" in cats, "Missing drift category in DQ summary."

[Postwrite] dq_summary rows: 14
[Postwrite] dq_nulls_by_column rows: 23
+--------------+-------------------------------+-------------------+--------------------+----+-----+--------------------------+
|category      |metric_name                    |metric_value       |metric_rate         |year|month|run_timestamp             |
+--------------+-------------------------------+-------------------+--------------------+----+-----+--------------------------+
|drift         |extra_silver_columns           |15.0               |NULL                |2024|1    |2026-02-24 11:44:12.028073|
|drift         |missing_expected_silver_columns|0.0                |NULL                |2024|1    |2026-02-24 11:44:12.028073|
|reconciliation|bronze_rows                    |2964624.0          |NULL                |2024|1    |2026-02-24 11:44:12.028073|
|reconciliation|bronze_to_silver_drop_rate     |0.04224549217708552|NULL                |2024|1    |2026-02-24 11:44:12.028073|
|reconciliation|bronze_to_silver